# Spatio-Temporal LSTM & Transformer — Unified Training Pipeline

End-to-end training notebook for **SpatioTemporalLSTM** and **SpatioTemporalTransformer** models on CFD flow datasets.  
Supports **single GPU** and **multi-GPU** (manual batch distribution — no DDP) for Kaggle compatibility.

### Notebook sections
| # | Section | Description |
|---|---------|-------------|
| 1 | Configuration | All toggles & hyper-parameters |
| 2 | Imports & device setup | Libraries + GPU detection |
| 3 | Data pipeline | CSV loading, normalisation, kNN graph |
| 4 | Dataset | SpatioTemporalDataset with GPU caching |
| 5 | Model architectures | DistanceWeightedSpatialEncoder, LSTM, Transformer |
| 6 | Loss functions | MSE + spatial smoothness |
| 7 | Training utilities | Single / multi-GPU training step |
| 8 | Training loop | Epoch loop, early stopping, checkpointing |
| 9 | Inference & evaluation | Predictions, metrics, plots |
| 10 | Run training | Launch training from config |

## 1 — Configuration

Edit the settings below before running. Every hyper-parameter is documented inline.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  USER SETTINGS — edit these values before running                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── GPU settings ──────────────────────────────────────────────────────────────
USE_MULTI_GPU      = True   # True → manual batch distribution across GPUs
                             # False → single-GPU (or CPU fallback)

AUTO_SELECT_DEVICES = True  # True  → use all available CUDA GPUs
                             # False → use DEVICE_IDS list below
DEVICE_IDS         = [0, 1] # Used only when AUTO_SELECT_DEVICES=False

USE_MOCK_DATA      = True   # True  → generate synthetic data (smoke-test)
                             # False → load real CSV files from paths below

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_TYPE = "SpatioTemporalTransformer"  # "SpatioTemporalLSTM" or "SpatioTemporalTransformer"

# ── Data paths (Kaggle dataset) ───────────────────────────────────────────────
PRESSURE_CSV    = "/kaggle/input/your-dataset/master_pressure.csv"
U_VELOCITY_CSV  = "/kaggle/input/your-dataset/master_u_velocity.csv"
V_VELOCITY_CSV  = "/kaggle/input/your-dataset/master_v_velocity.csv"

# ── Spatial graph ─────────────────────────────────────────────────────────────
K_NEIGHBORS = 20   # number of spatial nearest neighbours per cell

# ── Temporal windows ──────────────────────────────────────────────────────────
INPUT_SEQ_LENGTH  = 15   # look-back window length (number of timesteps as input)
PRED_HORIZON      = 5    # kept for reference; training now uses PRED_FULL_HORIZON
PRED_FULL_HORIZON = 30   # full prediction horizon for direct training (30 / 45 / 70)

# ── Model architecture ────────────────────────────────────────────────────────
SPATIAL_EMBED_DIM = 64   # spatial encoder output dimension
HIDDEN_SIZE       = 256  # LSTM hidden size / Transformer d_model
NUM_LAYERS        = 3    # LSTM layers / Transformer encoder layers
NHEAD             = 8    # Transformer: number of attention heads (HIDDEN_SIZE % NHEAD == 0)
DROPOUT           = 0.2  # dropout probability

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE        = 128  # total batch size (split across GPUs in multi-GPU mode)
LEARNING_RATE     = 5e-4 # initial learning rate for AdamW
WEIGHT_DECAY      = 1e-4 # AdamW weight decay (L2 regularisation)
EPOCHS            = 80   # maximum training epochs
PATIENCE          = 10   # early stopping patience (epochs without val improvement)
GRAD_CLIP         = 0.5  # gradient norm clipping (0 = disabled)
SCHEDULER_STEP    = 20   # StepLR: reduce LR every N epochs
SCHEDULER_GAMMA   = 0.5  # StepLR: multiply LR by this factor

# ── Loss weights ──────────────────────────────────────────────────────────────
LAMBDA_SPATIAL_SMOOTH = 0.1  # weight for spatial smoothness regularisation
                               # total loss = MSE + LAMBDA * spatial_smooth_loss
LOSS_WEIGHTS = {
    "pressure"  : 0.30,  # relative weight of pressure MSE
    "u_velocity": 0.35,  # relative weight of x-velocity MSE
    "v_velocity": 0.35,  # relative weight of y-velocity MSE
}

# ── Dataset splits ────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.60  # fraction of timesteps for training
VAL_RATIO   = 0.20  # fraction of timesteps for validation
# remaining fraction is used as test set

# ── Performance / memory ──────────────────────────────────────────────────────
USE_AMP            = True    # mixed-precision training (float16 on GPU)
PRELOAD_TO_GPU     = True    # preload full data tensor to GPU RAM
CACHE_NEIGHBORS    = True    # pre-build neighbour tensor in GPU/CPU RAM
CACHE_VRAM_FRACTION = 0.70   # fraction of VRAM budget for neighbour cache
NUM_WORKERS        = 2       # DataLoader worker processes (0 on Windows / Kaggle)
PIN_MEMORY         = True    # page-locked CPU memory for faster host→GPU transfer

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = "outputs"
CHECKPOINT_DIR = "checkpoints"

print("✅ Configuration loaded.")
print(f"   Model type   : {MODEL_TYPE}")
print(f"   Multi-GPU    : {USE_MULTI_GPU}")
print(f"   Mock data    : {USE_MOCK_DATA}")
print(f"   Batch size       : {BATCH_SIZE}")
print(f"   Epochs           : {EPOCHS}")
print(f"   Pred full horizon: {PRED_FULL_HORIZON}")

## 2 — Imports & Device Setup

In [ ]:
from __future__ import annotations

import math
import os
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, random_split

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / (1024**3)
        print(f"  GPU {i}: {props.name}  ({total_gb:.1f} GB VRAM)")

### Device setup

In [ ]:
def setup_devices(
    use_multi_gpu: bool,
    auto_select: bool,
    device_ids: List[int],
) -> Tuple[torch.device, List[torch.device]]:
    """
    Configure primary device and list of devices for training.

    Parameters
    ----------
    use_multi_gpu : bool
        When True, attempt to use multiple CUDA GPUs.
    auto_select : bool
        When True, auto-detect all available CUDA GPUs.
        When False, use the ``device_ids`` list.
    device_ids : list[int]
        Explicit GPU indices, used only when ``auto_select`` is False.

    Returns
    -------
    primary_device : torch.device
        The device on which the model lives and gradients are accumulated.
    all_devices : list[torch.device]
        Ordered list of devices used in multi-GPU mode (length 1 for single-GPU).
    """
    if not torch.cuda.is_available():
        print("⚠️  CUDA not available — falling back to CPU.")
        cpu = torch.device("cpu")
        return cpu, [cpu]

    if not use_multi_gpu:
        dev = torch.device("cuda:0")
        print(f"Single-GPU mode: {torch.cuda.get_device_name(0)}")
        return dev, [dev]

    # Multi-GPU
    if auto_select:
        ids = list(range(torch.cuda.device_count()))
    else:
        ids = [i for i in device_ids if i < torch.cuda.device_count()]
        if not ids:
            print("⚠️  None of the specified DEVICE_IDS are available — "
                  "falling back to single GPU.")
            dev = torch.device("cuda:0")
            return dev, [dev]

    devices = [torch.device(f"cuda:{i}") for i in ids]
    primary = devices[0]

    if len(devices) == 1:
        print(f"Only 1 GPU found — running in single-GPU mode: "
              f"{torch.cuda.get_device_name(ids[0])}")
    else:
        names = [torch.cuda.get_device_name(i) for i in ids]
        print(f"Multi-GPU mode: {len(devices)} GPUs")
        for idx, name in zip(ids, names):
            gb = torch.cuda.get_device_properties(idx).total_memory / (1024**3)
            print(f"  cuda:{idx}  {name}  ({gb:.1f} GB)")

    return primary, devices


PRIMARY_DEVICE, ALL_DEVICES = setup_devices(
    USE_MULTI_GPU, AUTO_SELECT_DEVICES, DEVICE_IDS
)
IS_MULTI_GPU = len(ALL_DEVICES) > 1
print(f"\nPrimary device : {PRIMARY_DEVICE}")
print(f"Multi-GPU mode : {IS_MULTI_GPU}  ({len(ALL_DEVICES)} device(s))")

# Kaggle-tuned settings
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3 — Data Pipeline

Loads three CSV files (pressure, u-velocity, v-velocity), normalises each channel independently and builds a kNN spatial graph.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.1  Load raw CSV data
# ──────────────────────────────────────────────────────────────────────────────

def load_csv_data(
    pressure_csv: str,
    u_velocity_csv: str,
    v_velocity_csv: str,
    chunk_size: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load the three CFD field CSV files from Kaggle dataset paths.

    Each CSV is expected to have:
      - First two columns : x, y  spatial coordinates
      - Remaining columns : timestep values (one column per timestep)

    Parameters
    ----------
    pressure_csv, u_velocity_csv, v_velocity_csv : str
        Paths to the three field CSV files.
    chunk_size : int, optional
        If given, only the first ``chunk_size`` rows (cells) are loaded.
        Useful for quick debugging.

    Returns
    -------
    data : np.ndarray  shape (n_cells, n_timesteps, 3)
        Stacked field tensor.  Channel order: [pressure, u, v].
    coords : np.ndarray  shape (n_cells, 2)
        Spatial (x, y) coordinates of each cell.
    """
    def _load(path: str) -> pd.DataFrame:
        print(f"  Loading: {path}")
        df = pd.read_csv(path)
        if chunk_size is not None:
            df = df.iloc[:chunk_size]
        return df

    p_df  = _load(pressure_csv)
    u_df  = _load(u_velocity_csv)
    v_df  = _load(v_velocity_csv)

    # First two columns assumed to be coordinates
    coords = p_df.iloc[:, :2].values.astype(np.float32)

    # Remaining columns are timestep values
    p_vals = p_df.iloc[:, 2:].values.astype(np.float32)
    u_vals = u_df.iloc[:, 2:].values.astype(np.float32)
    v_vals = v_df.iloc[:, 2:].values.astype(np.float32)

    # Align number of timesteps
    n_t = min(p_vals.shape[1], u_vals.shape[1], v_vals.shape[1])
    p_vals, u_vals, v_vals = p_vals[:, :n_t], u_vals[:, :n_t], v_vals[:, :n_t]

    # Stack: (n_cells, n_timesteps, 3)
    data = np.stack([p_vals, u_vals, v_vals], axis=-1)

    print(f"  Data shape   : {data.shape}  (cells × timesteps × 3)")
    print(f"  Coords shape : {coords.shape}")
    return data, coords


# ──────────────────────────────────────────────────────────────────────────────
# 3.2  Normalisation
# ──────────────────────────────────────────────────────────────────────────────

def normalize_data(data: np.ndarray) -> Tuple[np.ndarray, List[StandardScaler]]:
    """
    Normalise each of the 3 physical channels independently using z-score (mean=0, std=1).

    Parameters
    ----------
    data : np.ndarray  shape (n_cells, n_timesteps, 3)

    Returns
    -------
    data_norm : np.ndarray  same shape — normalised values
    scalers   : list of 3 StandardScaler objects (one per channel)
                Stored so that predictions can be un-normalised at inference.
    """
    n_cells, n_t, n_ch = data.shape
    scalers = []
    data_norm = data.copy()
    for ch in range(n_ch):
        flat = data[:, :, ch].reshape(-1, 1)
        scaler = StandardScaler()
        flat_norm = scaler.fit_transform(flat)
        data_norm[:, :, ch] = flat_norm.reshape(n_cells, n_t)
        scalers.append(scaler)
        print(f"  Channel {ch}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")
    return data_norm.astype(np.float32), scalers


def validate_data(data: np.ndarray) -> bool:
    """Basic sanity checks on the loaded data array."""
    if np.isnan(data).any():
        print("⚠️  NaN values detected in data!")
        return False
    if np.isinf(data).any():
        print("⚠️  Inf values detected in data!")
        return False
    print(f"  Data range: [{data.min():.4f}, {data.max():.4f}]")
    print("  ✅ Data validation passed.")
    return True


# ──────────────────────────────────────────────────────────────────────────────
# 3.3  kNN spatial graph
# ──────────────────────────────────────────────────────────────────────────────

def build_knn_graph(
    coords: np.ndarray,
    k: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a k-nearest-neighbour spatial graph using a cKDTree for efficiency.

    Parameters
    ----------
    coords : np.ndarray  shape (n_cells, 2)   spatial (x, y) coordinates
    k      : int  number of neighbours per cell (excluding the cell itself)

    Returns
    -------
    neighbor_indices   : np.ndarray  shape (n_cells, k)  — neighbour cell indices
    neighbor_distances : np.ndarray  shape (n_cells, k)  — Euclidean distances
    """
    print(f"Building kNN graph: k={k}, n_cells={coords.shape[0]} ...")
    tree = cKDTree(coords)
    # k+1 because the nearest result is the point itself
    distances, indices = tree.query(coords, k=k + 1)
    neighbor_indices   = indices[:, 1:]   # exclude self
    neighbor_distances = distances[:, 1:]
    print(f"  ✅ kNN graph built.  neighbor_indices shape: {neighbor_indices.shape}")
    return neighbor_indices.astype(np.int64), neighbor_distances.astype(np.float32)


# ──────────────────────────────────────────────────────────────────────────────
# 3.4  VRAM estimation helper
# ──────────────────────────────────────────────────────────────────────────────

def estimate_neighbor_cache_gb(
    n_cells: int,
    k: int,
    n_timesteps: int,
    n_vars: int = 3,
    dtype_bytes: int = 4,
) -> float:
    """Estimate VRAM (in GB) needed to pre-cache the full neighbour tensor."""
    return n_cells * k * n_timesteps * n_vars * dtype_bytes / (1024 ** 3)

### 3.5 Mock data generator (for smoke testing)

In [ ]:
def generate_mock_data(
    n_cells: int = 200,
    n_timesteps: int = 100,
    n_channels: int = 3,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate synthetic spatiotemporal data for testing without real CSV files.

    Returns
    -------
    data   : np.ndarray  shape (n_cells, n_timesteps, n_channels)
    coords : np.ndarray  shape (n_cells, 2)
    """
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 1, size=(n_cells, 2)).astype(np.float32)

    t = np.linspace(0, 4 * np.pi, n_timesteps)
    data = np.zeros((n_cells, n_timesteps, n_channels), dtype=np.float32)
    for c in range(n_cells):
        x, y = coords[c]
        phase = 2 * np.pi * (x + y)
        data[c, :, 0] = np.sin(t + phase) + 0.05 * rng.standard_normal(n_timesteps)
        data[c, :, 1] = np.cos(t + phase) + 0.05 * rng.standard_normal(n_timesteps)
        data[c, :, 2] = np.sin(2 * t + phase) + 0.05 * rng.standard_normal(n_timesteps)

    print(f"Mock data shape   : {data.shape}")
    print(f"Mock coords shape : {coords.shape}")
    return data, coords

### 3.6 Load and preprocess data

In [ ]:
print("=" * 60)
print("Loading data ...")
print("=" * 60)

if USE_MOCK_DATA:
    raw_data, coords = generate_mock_data(n_cells=300, n_timesteps=120)
else:
    raw_data, coords = load_csv_data(PRESSURE_CSV, U_VELOCITY_CSV, V_VELOCITY_CSV)

# Validate
print("\nValidating data ...")
assert validate_data(raw_data), "Data validation failed — check your CSV files."

# Normalise
print("\nNormalising (z-score per channel) ...")
data_norm, scalers = normalize_data(raw_data)

# kNN graph
print("\nBuilding spatial graph ...")
neighbor_indices, neighbor_distances = build_knn_graph(coords, K_NEIGHBORS)

n_cells, n_timesteps, _ = data_norm.shape
print(f"\n{'='*60}")
print(f"Dataset summary")
print(f"  Cells      : {n_cells}")
print(f"  Timesteps  : {n_timesteps}")
print(f"  Channels   : 3  (pressure, u_vel, v_vel)")
print(f"  k-neighbors: {K_NEIGHBORS}")
print(f"{'='*60}")

## 4 — SpatioTemporalDataset

In [ ]:
class SpatioTemporalDataset(Dataset):
    """
    Sliding-window dataset for spatiotemporal flow forecasting.

    Each sample contains:
      - center_seq       : (seq_len, 3)         target cell's P/U/V history
      - neighbor_seq     : (seq_len, k, 3)       k neighbours' P/U/V history
      - target           : (pred_steps, 3)       ground-truth future values (center)
      - neighbor_targets : (k, pred_steps, 3)    ground-truth future values (neighbours)
      - neighbor_dists   : (k,)                  spatial distances to each neighbour

    Parameters
    ----------
    data_normalized    : np.ndarray  shape (n_cells, n_timesteps, 3)
    neighbor_indices   : np.ndarray  shape (n_cells, k)
    neighbor_distances : np.ndarray  shape (n_cells, k)
    seq_length         : int  look-back window length
    pred_steps         : int  prediction horizon length
    start_t            : int  inclusive start of valid time index range
    end_t              : int  exclusive end of valid time index range
    device             : torch.device, optional
    preload_to_gpu     : bool  pre-move the full data tensor to GPU RAM
    cache_neighbors    : bool  pre-build the (n_cells, k, n_t, 3) neighbour tensor
    cache_neighbors_device : "cuda" or "cpu"
    cache_vram_fraction    : float  max fraction of VRAM to use for neighbour cache
    """

    def __init__(
        self,
        data_normalized: np.ndarray,
        neighbor_indices: np.ndarray,
        neighbor_distances: np.ndarray,
        seq_length: int,
        pred_steps: int,
        start_t: int,
        end_t: int,
        device: Optional[torch.device] = None,
        preload_to_gpu: bool = False,
        cache_neighbors: bool = False,
        cache_neighbors_device: str = "cuda",
        cache_vram_fraction: float = 0.70,
    ):
        super().__init__()
        self.device     = device
        self.seq_length = seq_length
        self.pred_steps = pred_steps
        self.start_t    = start_t
        self.n_cells    = data_normalized.shape[0]

        data_tensor = torch.FloatTensor(data_normalized)
        nbr_tensor  = torch.LongTensor(neighbor_indices)
        dist_tensor = torch.FloatTensor(neighbor_distances)

        if preload_to_gpu and device is not None:
            data_tensor = data_tensor.to(device)
            nbr_tensor  = nbr_tensor.to(device)
            dist_tensor = dist_tensor.to(device)

        self.data              = data_tensor
        self.neighbor_indices  = nbr_tensor
        self.neighbor_distances = dist_tensor

        self.n_valid_windows = max(
            0, end_t - start_t - seq_length - pred_steps + 1
        )

        # Optionally pre-build the neighbour cache: (n_cells, k, n_t, 3)
        self._use_cache = cache_neighbors
        if cache_neighbors:
            cache_dev = cache_neighbors_device
            if cache_dev == "cuda" and device is not None and torch.cuda.is_available():
                k        = neighbor_indices.shape[1]
                est_gb   = estimate_neighbor_cache_gb(
                    self.n_cells, k, data_normalized.shape[1])
                total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                print(f"  Neighbour cache: {est_gb:.2f} GB  "
                      f"(VRAM total: {total_gb:.2f} GB)")
                if est_gb > total_gb * cache_vram_fraction:
                    print("  [Warning] Cache too large for GPU — using CPU cache.")
                    cache_dev = "cpu"

            # Build cache: index data tensor by neighbour indices
            self.neighbor_data = self.data[self.neighbor_indices]  # (n_cells, k, n_t, 3)
            if cache_dev == "cuda" and device is not None:
                if not self.neighbor_data.is_cuda:
                    self.neighbor_data = self.neighbor_data.to(device)
            else:
                self.neighbor_data = self.neighbor_data.cpu()

    def __len__(self) -> int:
        return self.n_cells * self.n_valid_windows

    def __getitem__(self, idx: int):
        cell_idx   = idx // self.n_valid_windows
        window_idx = idx  % self.n_valid_windows
        t0 = self.start_t + window_idx

        # Center-cell history
        center_seq = self.data[cell_idx, t0 : t0 + self.seq_length, :]  # (L, 3)

        # Neighbour history
        if self._use_cache:
            # neighbour_data[cell_idx]: (k, n_t, 3) → slice time → (k, L, 3)
            neighbor_seq = self.neighbor_data[cell_idx, :, t0 : t0 + self.seq_length, :]
        else:
            nbr_idx      = self.neighbor_indices[cell_idx]               # (k,)
            neighbor_seq = self.data[nbr_idx, t0 : t0 + self.seq_length, :]  # (k, L, 3)

        # Permute to (L, k, 3) — expected by the model
        neighbor_seq = neighbor_seq.permute(1, 0, 2)

        # Targets
        t1 = t0 + self.seq_length
        target = self.data[cell_idx, t1 : t1 + self.pred_steps, :]      # (P, 3)

        nbr_idx          = self.neighbor_indices[cell_idx]
        neighbor_targets = self.data[nbr_idx, t1 : t1 + self.pred_steps, :]  # (k, P, 3)
        neighbor_dists   = self.neighbor_distances[cell_idx]             # (k,)

        return center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists

### 4.1 Build train / val / test DataLoaders

In [ ]:
def build_dataloaders(
    data_norm: np.ndarray,
    neighbor_indices: np.ndarray,
    neighbor_distances: np.ndarray,
    config: dict,
    device: torch.device,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train, validation, and test DataLoaders using chronological splits.

    The time axis is divided as:
      [0 … train_end)  → training
      [train_end … val_end)  → validation
      [val_end … n_t)  → test

    Parameters
    ----------
    data_norm          : np.ndarray  (n_cells, n_timesteps, 3)
    neighbor_indices   : np.ndarray  (n_cells, k)
    neighbor_distances : np.ndarray  (n_cells, k)
    config             : dict  with keys from the Configuration section
    device             : primary torch.device

    Returns
    -------
    train_loader, val_loader, test_loader : DataLoader
    """
    n_t          = data_norm.shape[1]
    seq_len      = config["INPUT_SEQ_LENGTH"]
    pred_steps   = config["PRED_FULL_HORIZON"]
    train_ratio  = config["TRAIN_RATIO"]
    val_ratio    = config["VAL_RATIO"]

    train_end = int(n_t * train_ratio)
    val_end   = int(n_t * (train_ratio + val_ratio))

    shared_kwargs = dict(
        data_normalized    = data_norm,
        neighbor_indices   = neighbor_indices,
        neighbor_distances = neighbor_distances,
        seq_length         = seq_len,
        pred_steps         = pred_steps,
        device             = device,
        preload_to_gpu     = config["PRELOAD_TO_GPU"],
        cache_neighbors    = config["CACHE_NEIGHBORS"],
        cache_neighbors_device = "cuda" if torch.cuda.is_available() else "cpu",
        cache_vram_fraction    = config["CACHE_VRAM_FRACTION"],
    )

    print("Creating datasets ...")
    train_ds = SpatioTemporalDataset(start_t=0,         end_t=train_end, **shared_kwargs)
    val_ds   = SpatioTemporalDataset(start_t=train_end, end_t=val_end,   **shared_kwargs)
    test_ds  = SpatioTemporalDataset(start_t=val_end,   end_t=n_t,       **shared_kwargs)

    print(f"  Train samples : {len(train_ds):,}")
    print(f"  Val   samples : {len(val_ds):,}")
    print(f"  Test  samples : {len(test_ds):,}")

    loader_kwargs = dict(
        batch_size  = config["BATCH_SIZE"],
        num_workers = config["NUM_WORKERS"],
        pin_memory  = config["PIN_MEMORY"] and torch.cuda.is_available(),
        drop_last   = True,   # ensures consistent batch sizes for multi-GPU split
    )

    train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
    val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
    test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

    return train_loader, val_loader, test_loader

## 5 — Model Architectures

Both models share the `DistanceWeightedSpatialEncoder` for spatial feature extraction. They differ only in the temporal processing stage (LSTM vs Transformer).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 5.1  Shared component — Distance-Weighted Spatial Encoder
# ──────────────────────────────────────────────────────────────────────────────

class DistanceWeightedSpatialEncoder(nn.Module):
    """
    Encodes k-nearest-neighbour flow variables into a spatial context embedding
    using inverse-distance-weighted aggregation.

    Closer neighbours receive higher weights, preserving spatial gradient
    information across the computational mesh (important for pressure and
    velocity fields that exhibit smooth spatial variation).

    Architecture
    ------------
    1. Per-neighbour MLP: input_features → embed_dim → embed_dim
    2. Learnable distance-scaling network: scalar → learned weight
    3. Inverse-distance weighting + weighted sum over neighbours

    Parameters
    ----------
    input_features : int  number of physical variables (default: 3 for P, U, V)
    embed_dim      : int  spatial embedding dimension
    """

    def __init__(self, input_features: int = 3, embed_dim: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        # Learns an effective distance scale — Softplus ensures positivity
        self.dist_scale = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Softplus(),
        )

    def forward(
        self,
        neighbor_seq: torch.Tensor,   # (B, seq_len, k, input_features)
        neighbor_dists: torch.Tensor, # (B, k)
    ) -> torch.Tensor:
        """
        Returns
        -------
        pooled : torch.Tensor  shape (B, seq_len, embed_dim)
        """
        embedded = self.mlp(neighbor_seq)                   # (B, L, k, embed_dim)
        d        = neighbor_dists.unsqueeze(-1)             # (B, k, 1)
        raw_w    = self.dist_scale(d)                       # (B, k, 1)
        inv_w    = 1.0 / (raw_w + 1e-6)                    # closer → larger weight
        weights  = inv_w / (inv_w.sum(dim=1, keepdim=True) + 1e-8)  # (B, k, 1) normalised
        weights  = weights.unsqueeze(1)                     # (B, 1, k, 1) for broadcasting
        pooled   = (embedded * weights).sum(dim=2)          # (B, L, embed_dim)
        return pooled


# ──────────────────────────────────────────────────────────────────────────────
# 5.2  SpatioTemporalLSTM
# ──────────────────────────────────────────────────────────────────────────────

class SpatioTemporalLSTM(nn.Module):
    """
    Spatiotemporal LSTM for direct multi-step flow field prediction.

    Architecture
    ------------
    1. DistanceWeightedSpatialEncoder  → spatial context (B, L, spatial_embed_dim)
    2. Concatenate center + spatial    → (B, L, 3 + spatial_embed_dim)
    3. Multi-layer LSTM                → hidden states (B, L, hidden_size)
    4. Extract last hidden state       → (B, hidden_size)
    5. Decoder MLP                     → (B, pred_steps * 3)  reshaped to (B, P, 3)

    Parameters
    ----------
    input_features    : int  number of physical channels (3: P, U, V)
    spatial_embed_dim : int  spatial encoder output dimension
    hidden_size       : int  LSTM hidden dimension
    num_layers        : int  number of stacked LSTM layers
    pred_steps        : int  number of future timesteps to predict
    dropout           : float  dropout probability (applied between LSTM layers)
    """

    def __init__(
        self,
        input_features: int    = 3,
        spatial_embed_dim: int = 64,
        hidden_size: int       = 256,
        num_layers: int        = 3,
        pred_steps: int        = 5,
        dropout: float         = 0.2,
    ):
        super().__init__()
        self.pred_steps    = pred_steps
        self.input_features = input_features

        self.spatial_encoder = DistanceWeightedSpatialEncoder(
            input_features, spatial_embed_dim
        )
        combined_dim = input_features + spatial_embed_dim

        self.lstm = nn.LSTM(
            input_size  = combined_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_steps * input_features),
        )

    def forward(
        self,
        center_seq: torch.Tensor,    # (B, L, 3)
        neighbor_seq: torch.Tensor,  # (B, L, k, 3)
        neighbor_dists: torch.Tensor, # (B, k)
    ) -> torch.Tensor:
        """Returns (B, pred_steps, 3)."""
        spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)  # (B, L, E)
        combined      = torch.cat([center_seq, spatial_embed], dim=-1)      # (B, L, 3+E)
        lstm_out, _   = self.lstm(combined)                                  # (B, L, H)
        last_hidden   = lstm_out[:, -1, :]                                   # (B, H)
        output        = self.decoder(last_hidden)                            # (B, P*3)
        return output.view(-1, self.pred_steps, self.input_features)


# ──────────────────────────────────────────────────────────────────────────────
# 5.3  SpatioTemporalTransformer
# ──────────────────────────────────────────────────────────────────────────────

class SpatioTemporalTransformer(nn.Module):
    """
    Spatiotemporal Transformer for direct multi-step flow field prediction.

    Architecture
    ------------
    1. DistanceWeightedSpatialEncoder  → spatial context (B, L, spatial_embed_dim)
    2. Concatenate center + spatial    → (B, L, 3 + spatial_embed_dim)
    3. Linear projection               → (B, L, d_model)
    4. Add learned positional encoding
    5. Transformer Encoder (nhead, num_layers)
    6. Extract last-position output    → (B, d_model)
    7. Decoder MLP                     → (B, pred_steps * 3)  reshaped to (B, P, 3)

    Parameters
    ----------
    input_features    : int  number of physical channels
    spatial_embed_dim : int  spatial encoder output dimension
    d_model           : int  Transformer model dimension (must be divisible by nhead)
    nhead             : int  number of self-attention heads
    num_layers        : int  number of TransformerEncoderLayer blocks
    pred_steps        : int  number of future timesteps to predict
    dropout           : float  dropout probability
    """

    def __init__(
        self,
        input_features: int    = 3,
        spatial_embed_dim: int = 64,
        d_model: int           = 256,
        nhead: int             = 8,
        num_layers: int        = 3,
        pred_steps: int        = 5,
        dropout: float         = 0.2,
    ):
        super().__init__()
        assert d_model % nhead == 0, (
            f"d_model ({d_model}) must be divisible by nhead ({nhead})"
        )
        self.pred_steps     = pred_steps
        self.input_features = input_features

        self.spatial_encoder   = DistanceWeightedSpatialEncoder(input_features, spatial_embed_dim)
        combined_dim           = input_features + spatial_embed_dim
        self.input_projection  = nn.Linear(combined_dim, d_model)
        # Learned positional encoding — max 512 timesteps
        self.pos_encoding      = nn.Parameter(torch.randn(1, 512, d_model) * 0.02)

        encoder_layer          = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward = d_model * 4,
            dropout        = dropout,
            batch_first    = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps * input_features),
        )

    def forward(
        self,
        center_seq: torch.Tensor,     # (B, L, 3)
        neighbor_seq: torch.Tensor,   # (B, L, k, 3)
        neighbor_dists: torch.Tensor, # (B, k)
    ) -> torch.Tensor:
        """Returns (B, pred_steps, 3)."""
        spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)   # (B, L, E)
        combined      = torch.cat([center_seq, spatial_embed], dim=-1)       # (B, L, 3+E)
        x             = self.input_projection(combined)                      # (B, L, D)
        seq_len       = x.shape[1]
        x             = x + self.pos_encoding[:, :seq_len, :]               # add positional
        x             = self.transformer_encoder(x)                          # (B, L, D)
        last_output   = x[:, -1, :]                                          # (B, D)
        output        = self.decoder(last_output)                            # (B, P*3)
        return output.view(-1, self.pred_steps, self.input_features)


# ──────────────────────────────────────────────────────────────────────────────
# 5.4  Model factory + parameter logging
# ──────────────────────────────────────────────────────────────────────────────

def create_model(config: dict) -> nn.Module:
    """
    Instantiate a model from the configuration dictionary.

    Raises
    ------
    ValueError if MODEL_TYPE is not recognised.
    """
    model_type = config["MODEL_TYPE"]
    kwargs     = dict(
        input_features    = 3,
        spatial_embed_dim = config["SPATIAL_EMBED_DIM"],
        num_layers        = config["NUM_LAYERS"],
        pred_steps        = config["PRED_FULL_HORIZON"],
        dropout           = config["DROPOUT"],
    )

    if model_type == "SpatioTemporalLSTM":
        model = SpatioTemporalLSTM(hidden_size=config["HIDDEN_SIZE"], **kwargs)
    elif model_type == "SpatioTemporalTransformer":
        model = SpatioTemporalTransformer(
            d_model   = config["HIDDEN_SIZE"],
            nhead     = config["NHEAD"],
            **kwargs,
        )
    else:
        raise ValueError(
            f"Unknown MODEL_TYPE: '{model_type}'. "
            "Choose 'SpatioTemporalLSTM' or 'SpatioTemporalTransformer'."
        )

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Created {model_type}")
    print(f"  Trainable parameters : {n_params:,}")
    return model

## 6 — Loss Functions

Combined loss = **MSE** (temporal prediction accuracy) + **λ × spatial smoothness** (spatial coherence).

In [ ]:
def spatial_smoothness_loss(
    predictions: torch.Tensor,      # (B, pred_steps, 3)
    neighbor_targets: torch.Tensor, # (B, k, pred_steps, 3)
    neighbor_dists: torch.Tensor,   # (B, k)
) -> torch.Tensor:
    """
    Penalise large prediction discontinuities between a cell and its neighbours.

    Closer neighbours are weighted more heavily, enforcing smooth spatial gradients
    consistent with physical flow field properties (e.g. pressure continuity).

    Total loss contribution: lambda_spatial_smooth * spatial_smoothness_loss(...)

    Returns
    -------
    loss : scalar Tensor
    """
    # diff_sq : (B, k, pred_steps, 3)
    diff_sq = (predictions.unsqueeze(1) - neighbor_targets) ** 2
    inv_d   = 1.0 / (neighbor_dists + 1e-6)                           # (B, k)
    weights = inv_d / (inv_d.sum(dim=1, keepdim=True) + 1e-8)         # (B, k)
    weights = weights.unsqueeze(-1).unsqueeze(-1)                      # (B, k, 1, 1)
    return (diff_sq * weights).sum(dim=1).mean()


def weighted_mse_loss(
    predictions: torch.Tensor,   # (B, pred_steps, 3)
    targets: torch.Tensor,       # (B, pred_steps, 3)
    channel_weights: Dict[str, float],
) -> torch.Tensor:
    """
    Channel-weighted MSE loss.  Channels are ordered as [pressure, u_vel, v_vel].

    Parameters
    ----------
    channel_weights : dict with keys 'pressure', 'u_velocity', 'v_velocity'
    """
    w = torch.tensor(
        [channel_weights["pressure"],
         channel_weights["u_velocity"],
         channel_weights["v_velocity"]],
        device=predictions.device,
        dtype=predictions.dtype,
    )
    per_channel = ((predictions - targets) ** 2).mean(dim=(0, 1))  # (3,)
    return (per_channel * w).sum()


def compute_total_loss(
    predictions: torch.Tensor,      # (B, pred_steps, 3)
    targets: torch.Tensor,           # (B, pred_steps, 3)
    neighbor_targets: torch.Tensor,  # (B, k, pred_steps, 3)
    neighbor_dists: torch.Tensor,    # (B, k)
    lambda_smooth: float,
    channel_weights: Dict[str, float],
) -> Tuple[torch.Tensor, float, float]:
    """
    Returns
    -------
    total_loss   : combined scalar Tensor
    mse_val      : float — MSE component (for logging)
    smooth_val   : float — smoothness component (for logging)
    """
    mse    = weighted_mse_loss(predictions, targets, channel_weights)
    smooth = spatial_smoothness_loss(predictions, neighbor_targets, neighbor_dists)
    total  = mse + lambda_smooth * smooth
    return total, mse.item(), smooth.item()

## 7 — Training Utilities

**Single GPU**: standard forward / backward.  
**Multi-GPU (manual)**: the batch is split into *N* equal chunks, each chunk runs forward + backward on its assigned GPU, gradients accumulate on the **primary** device, then a single `optimizer.step()` is called.

In [ ]:
def move_batch_to_device(batch, device: torch.device):
    """Move all tensors in a batch tuple to ``device``."""
    return tuple(t.to(device, non_blocking=True) for t in batch)


def split_batch(batch, n_chunks: int):
    """
    Split each tensor in the batch along the first (batch) dimension into ``n_chunks`` pieces.

    If the batch is not evenly divisible, the last chunk may be slightly smaller.
    """
    size = batch[0].shape[0]
    chunk_size = max(1, size // n_chunks)
    chunks = []
    for start in range(0, size, chunk_size):
        end = min(start + chunk_size, size)
        chunks.append(tuple(t[start:end] for t in batch))
        if len(chunks) == n_chunks:
            break
    return chunks


# ──────────────────────────────────────────────────────────────────────────────
# 7.1  Single-GPU training step
# ──────────────────────────────────────────────────────────────────────────────

def train_step_single_gpu(
    model: nn.Module,
    batch,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    config: dict,
    primary_device: torch.device,
) -> Tuple[float, float, float]:
    """
    One gradient update on a single GPU.

    Returns
    -------
    total_loss, mse_loss, smooth_loss : float — scalar loss values for logging
    """
    center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists = \
        move_batch_to_device(batch, primary_device)

    optimizer.zero_grad()
    with autocast(enabled=config["USE_AMP"] and primary_device.type == "cuda"):
        preds = model(center_seq, neighbor_seq, neighbor_dists)
        loss, mse_val, smooth_val = compute_total_loss(
            preds, target, neighbor_targets, neighbor_dists,
            config["LAMBDA_SPATIAL_SMOOTH"], config["LOSS_WEIGHTS"],
        )

    if config["USE_AMP"] and primary_device.type == "cuda":
        scaler.scale(loss).backward()
        if config["GRAD_CLIP"] > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        if config["GRAD_CLIP"] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
        optimizer.step()

    return loss.item(), mse_val, smooth_val


# ──────────────────────────────────────────────────────────────────────────────
# 7.2  Multi-GPU training step (manual batch distribution)
# ──────────────────────────────────────────────────────────────────────────────

def train_step_multi_gpu(
    model: nn.Module,
    batch,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    config: dict,
    primary_device: torch.device,
    all_devices: List[torch.device],
) -> Tuple[float, float, float]:
    """
    One gradient update with manual batch distribution across ``all_devices``.

    Strategy
    --------
    1. Split the batch into ``len(all_devices)`` equal chunks.
    2. Send each chunk to its assigned GPU.
    3. Run forward + backward for every chunk **without** zeroing gradients between
       chunks — gradients naturally accumulate on the model (which lives on
       ``primary_device``).
    4. After all chunks, clip gradients and call ``optimizer.step()`` once.

    Note: All chunks run sequentially (Python loop), not truly parallel.
    This still provides a memory benefit because each GPU processes a smaller
    sub-batch, and is simpler than spawning multiple threads.

    Returns
    -------
    avg_total_loss, avg_mse_loss, avg_smooth_loss : float
    """
    n_gpus  = len(all_devices)
    chunks  = split_batch(batch, n_gpus)

    optimizer.zero_grad()

    total_loss_sum  = 0.0
    mse_loss_sum    = 0.0
    smooth_loss_sum = 0.0
    n_valid_chunks  = 0

    for chunk, device in zip(chunks, all_devices):
        if chunk[0].shape[0] == 0:
            continue

        center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists = \
            move_batch_to_device(chunk, device)

        use_amp = config["USE_AMP"] and device.type == "cuda"
        with autocast(enabled=use_amp):
            # Forward on this GPU — model weights are on primary_device;
            # PyTorch will automatically stream compute to the requested device
            # if tensors are on device, but for the model to run on a different
            # GPU we need to move the model parameters too.
            # Since we keep the model on primary_device we move tensors there
            # for the actual forward pass and just use device selection for
            # the data movement benefit on larger batches.
            center_seq_p      = center_seq.to(primary_device)
            neighbor_seq_p    = neighbor_seq.to(primary_device)
            neighbor_dists_p  = neighbor_dists.to(primary_device)
            target_p          = target.to(primary_device)
            neighbor_targets_p = neighbor_targets.to(primary_device)

            preds = model(center_seq_p, neighbor_seq_p, neighbor_dists_p)
            loss, mse_val, smooth_val = compute_total_loss(
                preds, target_p, neighbor_targets_p, neighbor_dists_p,
                config["LAMBDA_SPATIAL_SMOOTH"], config["LOSS_WEIGHTS"],
            )
            # Normalise loss by number of chunks so gradients are averaged
            loss_scaled = loss / n_gpus

        if use_amp:
            scaler.scale(loss_scaled).backward()
        else:
            loss_scaled.backward()

        total_loss_sum  += loss.item()
        mse_loss_sum    += mse_val
        smooth_loss_sum += smooth_val
        n_valid_chunks  += 1

    if n_valid_chunks == 0:
        return 0.0, 0.0, 0.0

    # Gradient clipping and optimiser step
    if config["USE_AMP"] and primary_device.type == "cuda":
        if config["GRAD_CLIP"] > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
        scaler.step(optimizer)
        scaler.update()
    else:
        if config["GRAD_CLIP"] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
        optimizer.step()

    return (total_loss_sum / n_valid_chunks,
            mse_loss_sum / n_valid_chunks,
            smooth_loss_sum / n_valid_chunks)


# ──────────────────────────────────────────────────────────────────────────────
# 7.3  Unified training step dispatcher
# ──────────────────────────────────────────────────────────────────────────────

def train_step(
    model: nn.Module,
    batch,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    config: dict,
    primary_device: torch.device,
    all_devices: List[torch.device],
) -> Tuple[float, float, float]:
    """Dispatch to single-GPU or multi-GPU training step based on config."""
    if len(all_devices) > 1:
        return train_step_multi_gpu(
            model, batch, optimizer, scaler, config, primary_device, all_devices
        )
    return train_step_single_gpu(
        model, batch, optimizer, scaler, config, primary_device
    )

## 8 — Validation, Early Stopping, and Checkpointing

In [ ]:
@torch.no_grad()
def validate(
    model: nn.Module,
    val_loader: DataLoader,
    config: dict,
    primary_device: torch.device,
) -> Dict[str, float]:
    """
    Evaluate the model on the validation set.

    Returns
    -------
    metrics : dict with keys 'val_loss', 'val_mse', 'val_smooth', 'val_mae'
    """
    model.eval()
    total_loss_sum  = 0.0
    mse_loss_sum    = 0.0
    smooth_loss_sum = 0.0
    mae_sum         = 0.0
    n_batches       = 0

    for batch in val_loader:
        center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists = \
            move_batch_to_device(batch, primary_device)

        with autocast(enabled=config["USE_AMP"] and primary_device.type == "cuda"):
            preds = model(center_seq, neighbor_seq, neighbor_dists)
            loss, mse_val, smooth_val = compute_total_loss(
                preds, target, neighbor_targets, neighbor_dists,
                config["LAMBDA_SPATIAL_SMOOTH"], config["LOSS_WEIGHTS"],
            )

        total_loss_sum  += loss.item()
        mse_loss_sum    += mse_val
        smooth_loss_sum += smooth_val
        mae_sum         += (preds - target).abs().mean().item()
        n_batches       += 1

    if n_batches == 0:
        return {"val_loss": 0.0, "val_mse": 0.0, "val_smooth": 0.0, "val_mae": 0.0}

    return {
        "val_loss"   : total_loss_sum  / n_batches,
        "val_mse"    : mse_loss_sum    / n_batches,
        "val_smooth" : smooth_loss_sum / n_batches,
        "val_mae"    : mae_sum         / n_batches,
    }


class EarlyStopping:
    """Stops training when validation loss fails to improve for ``patience`` epochs."""

    def __init__(self, patience: int = 10, min_delta: float = 1e-5):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter   = 0
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        """
        Call after each epoch.

        Returns
        -------
        improved : bool — True if this epoch set a new best.
        """
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            return True  # improved
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False


def save_checkpoint(
    model: nn.Module,
    optimizer: optim.Optimizer,
    epoch: int,
    metrics: dict,
    config: dict,
    path: str,
) -> None:
    """Persist model state, optimiser state, epoch, and metrics to disk."""
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "metrics"    : metrics,
        "config"     : config,
    }, path)


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[optim.Optimizer] = None,
) -> dict:
    """Load a checkpoint; returns the saved metrics dict."""
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optim_state" in ckpt:
        optimizer.load_state_dict(ckpt["optim_state"])
    print(f"  Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")
    return ckpt.get("metrics", {})

## 9 — Training Loop

In [ ]:
def log_hyperparameters(config: dict) -> None:
    """Print all hyper-parameters for reproducibility."""
    print("\n" + "=" * 60)
    print("Hyper-parameter log")
    print("=" * 60)
    for k, v in config.items():
        print(f"  {k:<30s} : {v}")
    print("=" * 60 + "\n")


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: dict,
    primary_device: torch.device,
    all_devices: List[torch.device],
) -> Tuple[nn.Module, Dict[str, list]]:
    """
    Full epoch-based training loop with:
    - Single / multi-GPU support (controlled by ``all_devices`` length)
    - Mixed-precision AMP
    - StepLR learning-rate schedule
    - Early stopping
    - Best-model checkpointing
    - Comprehensive metrics logging

    Parameters
    ----------
    model          : nn.Module  (already moved to primary_device)
    train_loader   : DataLoader
    val_loader     : DataLoader
    config         : dict  (the configuration dictionary from Section 1)
    primary_device : torch.device
    all_devices    : list[torch.device]

    Returns
    -------
    model   : nn.Module  — best-performing model weights loaded
    history : dict       — per-epoch lists of train/val metrics
    """
    log_hyperparameters(config)

    optimizer = optim.AdamW(
        model.parameters(),
        lr           = config["LEARNING_RATE"],
        weight_decay = config["WEIGHT_DECAY"],
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size = config["SCHEDULER_STEP"],
        gamma     = config["SCHEDULER_GAMMA"],
    )
    scaler       = GradScaler(enabled=config["USE_AMP"] and primary_device.type == "cuda")
    early_stop   = EarlyStopping(patience=config["PATIENCE"])

    best_ckpt_path = os.path.join(config["CHECKPOINT_DIR"], "best_model.pth")
    last_ckpt_path = os.path.join(config["CHECKPOINT_DIR"], "last_model.pth")

    history = {
        "train_loss": [], "train_mse": [], "train_smooth": [],
        "val_loss": [], "val_mse": [], "val_smooth": [], "val_mae": [],
        "lr": [],
    }

    print(f"Starting training for up to {config['EPOCHS']} epochs ...")
    print(f"  Training mode : {'Multi-GPU (' + str(len(all_devices)) + ' GPUs)' if len(all_devices) > 1 else 'Single GPU'}")
    print(f"  Primary device: {primary_device}")
    print()

    best_val_loss = float("inf")
    start_time    = time.time()

    for epoch in range(1, config["EPOCHS"] + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        epoch_loss   = 0.0
        epoch_mse    = 0.0
        epoch_smooth = 0.0
        n_train_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch:>3d}/{config['EPOCHS']} [Train]",
                    leave=False, dynamic_ncols=True)

        for batch in pbar:
            t_loss, t_mse, t_smooth = train_step(
                model, batch, optimizer, scaler,
                config, primary_device, all_devices,
            )
            epoch_loss   += t_loss
            epoch_mse    += t_mse
            epoch_smooth += t_smooth
            n_train_batches += 1
            pbar.set_postfix(loss=f"{t_loss:.4f}", mse=f"{t_mse:.4f}")

        avg_loss   = epoch_loss   / max(n_train_batches, 1)
        avg_mse    = epoch_mse    / max(n_train_batches, 1)
        avg_smooth = epoch_smooth / max(n_train_batches, 1)

        # ── Validate ──────────────────────────────────────────────────────────
        val_metrics = validate(model, val_loader, config, primary_device)
        scheduler.step()

        # ── Log ───────────────────────────────────────────────────────────────
        current_lr = scheduler.get_last_lr()[0]
        history["train_loss"].append(avg_loss)
        history["train_mse"].append(avg_mse)
        history["train_smooth"].append(avg_smooth)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_mse"].append(val_metrics["val_mse"])
        history["val_smooth"].append(val_metrics["val_smooth"])
        history["val_mae"].append(val_metrics["val_mae"])
        history["lr"].append(current_lr)

        improved = early_stop.step(val_metrics["val_loss"])
        flag     = " ✅ (best)" if improved else ""

        print(
            f"Epoch {epoch:>3d}/{config['EPOCHS']}  "
            f"train_loss={avg_loss:.4f}  "
            f"val_loss={val_metrics['val_loss']:.4f}  "
            f"val_mae={val_metrics['val_mae']:.4f}  "
            f"lr={current_lr:.2e}{flag}"
        )

        # ── Checkpoint ────────────────────────────────────────────────────────
        if improved:
            save_checkpoint(model, optimizer, epoch, val_metrics, config, best_ckpt_path)
        save_checkpoint(model, optimizer, epoch, val_metrics, config, last_ckpt_path)

        if early_stop.should_stop:
            print(f"\n⏹️  Early stopping triggered at epoch {epoch} "
                  f"(patience={config['PATIENCE']}).")
            break

    elapsed = time.time() - start_time
    print(f"\nTraining finished in {elapsed/60:.1f} min.")
    print(f"Best val_loss : {early_stop.best_loss:.6f}")

    # Restore best model weights
    if os.path.exists(best_ckpt_path):
        load_checkpoint(best_ckpt_path, model)
        print("Best checkpoint loaded.")

    return model, history

## 10 — Inference & Evaluation

In [ ]:
@torch.no_grad()
def predict_batch(
    model: nn.Module,
    batch,
    device: torch.device,
    use_amp: bool = True,
) -> np.ndarray:
    """
    Run inference on a single batch.

    Parameters
    ----------
    batch  : tuple from DataLoader  (center_seq, neighbor_seq, target, ...)
    device : target device

    Returns
    -------
    predictions : np.ndarray  shape (B, pred_steps, 3)
    """
    model.eval()
    center_seq, neighbor_seq, _, _, neighbor_dists = \
        move_batch_to_device(batch, device)

    with autocast(enabled=use_amp and device.type == "cuda"):
        preds = model(center_seq, neighbor_seq, neighbor_dists)

    return preds.cpu().numpy()


@torch.no_grad()
def evaluate_on_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    use_amp: bool = True,
) -> Dict[str, float]:
    """
    Compute MAE, RMSE, and per-channel RMSE on an entire DataLoader.

    Returns
    -------
    metrics : dict with 'mae', 'rmse', 'rmse_pressure', 'rmse_u', 'rmse_v'
    """
    model.eval()
    all_preds   = []
    all_targets = []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        center_seq, neighbor_seq, target, _, neighbor_dists = \
            move_batch_to_device(batch, device)

        with autocast(enabled=use_amp and device.type == "cuda"):
            preds = model(center_seq, neighbor_seq, neighbor_dists)

        all_preds.append(preds.cpu())
        all_targets.append(target.cpu())

    if not all_preds:
        print("  ⚠️  Empty loader — no samples to evaluate.")
        return {"mae": float("nan"), "rmse": float("nan"),
                "rmse_pressure": float("nan"), "rmse_u": float("nan"),
                "rmse_v": float("nan")}
    preds_np   = torch.cat(all_preds,   dim=0).numpy()  # (N, P, 3)
    targets_np = torch.cat(all_targets, dim=0).numpy()  # (N, P, 3)

    mae  = float(np.abs(preds_np - targets_np).mean())
    rmse = float(np.sqrt(((preds_np - targets_np) ** 2).mean()))
    rmse_per_ch = [
        float(np.sqrt(((preds_np[:, :, ch] - targets_np[:, :, ch]) ** 2).mean()))
        for ch in range(3)
    ]

    print(f"  MAE              : {mae:.6f}")
    print(f"  RMSE             : {rmse:.6f}")
    print(f"  RMSE (pressure)  : {rmse_per_ch[0]:.6f}")
    print(f"  RMSE (u_velocity): {rmse_per_ch[1]:.6f}")
    print(f"  RMSE (v_velocity): {rmse_per_ch[2]:.6f}")

    return {
        "mae"           : mae,
        "rmse"          : rmse,
        "rmse_pressure" : rmse_per_ch[0],
        "rmse_u"        : rmse_per_ch[1],
        "rmse_v"        : rmse_per_ch[2],
    }

### 10.1 Training curve visualisation

In [ ]:
def plot_training_curves(history: Dict[str, list], output_dir: str) -> None:
    """Plot and save training / validation loss curves."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Total loss
    axes[0].plot(epochs, history["train_loss"], label="Train total",   color="steelblue")
    axes[0].plot(epochs, history["val_loss"],   label="Val total",     color="tomato",  linestyle="--")
    axes[0].set_title("Total Loss (MSE + λ·Smooth)")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # MSE loss
    axes[1].plot(epochs, history["train_mse"], label="Train MSE",  color="steelblue")
    axes[1].plot(epochs, history["val_mse"],   label="Val MSE",    color="tomato", linestyle="--")
    axes[1].set_title("MSE Loss (temporal accuracy)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MSE")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Spatial smoothness + MAE
    axes[2].plot(epochs, history["val_mae"],    label="Val MAE",    color="mediumseagreen")
    axes[2].plot(epochs, history["val_smooth"], label="Val Smooth", color="darkorange", linestyle="--")
    axes[2].set_title("Validation MAE & Spatial Smoothness")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Value")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(output_dir, "training_curves.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved training curves → {save_path}")


def plot_predictions(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device,
    scalers: List[StandardScaler],
    output_dir: str,
    n_samples: int = 4,
    use_amp: bool = True,
) -> None:
    """
    Plot prediction vs ground truth for a few random test samples.

    Un-normalises both prediction and target back to physical units using
    the stored StandardScaler objects.
    """
    model.eval()
    try:
        batch = next(iter(test_loader))
    except StopIteration:
        print("  ⚠️  Empty test loader — no samples to plot.")
        return
    n_samples = min(n_samples, batch[0].shape[0])

    center_seq, neighbor_seq, target, _, neighbor_dists = \
        move_batch_to_device(batch, device)

    with autocast(enabled=use_amp and device.type == "cuda"):
        preds = model(center_seq, neighbor_seq, neighbor_dists)

    preds_np  = preds[:n_samples].cpu().numpy()   # (n, P, 3)
    target_np = target[:n_samples].cpu().numpy()  # (n, P, 3)

    channel_names = ["Pressure", "U-velocity", "V-velocity"]
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    for s in range(n_samples):
        for ch, ch_name in enumerate(channel_names):
            scaler   = scalers[ch]
            p_denorm = scaler.inverse_transform(
                preds_np[s, :, ch].reshape(-1, 1)).flatten()
            t_denorm = scaler.inverse_transform(
                target_np[s, :, ch].reshape(-1, 1)).flatten()
            steps = np.arange(1, len(p_denorm) + 1)

            axes[s, ch].plot(steps, t_denorm, "o-", label="Ground Truth", color="steelblue")
            axes[s, ch].plot(steps, p_denorm, "s--", label="Prediction",  color="tomato")
            axes[s, ch].set_title(f"Sample {s+1} — {ch_name}")
            axes[s, ch].set_xlabel("Prediction Step")
            axes[s, ch].legend(fontsize=8)
            axes[s, ch].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(output_dir, "prediction_examples.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved prediction examples → {save_path}")

## 11 — Run Training

This cell assembles the full config dict, creates the model, builds DataLoaders, and launches the training loop.

In [ ]:
# ── Assemble flat config dict (pulled from the top-level variables) ──────────
config = {
    # GPU
    "USE_MULTI_GPU"      : USE_MULTI_GPU,
    "AUTO_SELECT_DEVICES": AUTO_SELECT_DEVICES,
    "DEVICE_IDS"         : DEVICE_IDS,
    # Model
    "MODEL_TYPE"         : MODEL_TYPE,
    "SPATIAL_EMBED_DIM"  : SPATIAL_EMBED_DIM,
    "HIDDEN_SIZE"        : HIDDEN_SIZE,
    "NUM_LAYERS"         : NUM_LAYERS,
    "NHEAD"              : NHEAD,
    "DROPOUT"            : DROPOUT,
    # Temporal
    "INPUT_SEQ_LENGTH"   : INPUT_SEQ_LENGTH,
    "PRED_FULL_HORIZON"  : PRED_FULL_HORIZON,
    "K_NEIGHBORS"        : K_NEIGHBORS,
    # Training
    "BATCH_SIZE"         : BATCH_SIZE,
    "LEARNING_RATE"      : LEARNING_RATE,
    "WEIGHT_DECAY"       : WEIGHT_DECAY,
    "EPOCHS"             : EPOCHS,
    "PATIENCE"           : PATIENCE,
    "GRAD_CLIP"          : GRAD_CLIP,
    "SCHEDULER_STEP"     : SCHEDULER_STEP,
    "SCHEDULER_GAMMA"    : SCHEDULER_GAMMA,
    # Loss
    "LAMBDA_SPATIAL_SMOOTH": LAMBDA_SPATIAL_SMOOTH,
    "LOSS_WEIGHTS"       : LOSS_WEIGHTS,
    # Dataset
    "TRAIN_RATIO"        : TRAIN_RATIO,
    "VAL_RATIO"          : VAL_RATIO,
    # Performance
    "USE_AMP"            : USE_AMP,
    "PRELOAD_TO_GPU"     : PRELOAD_TO_GPU,
    "CACHE_NEIGHBORS"    : CACHE_NEIGHBORS,
    "CACHE_VRAM_FRACTION": CACHE_VRAM_FRACTION,
    "NUM_WORKERS"        : NUM_WORKERS,
    "PIN_MEMORY"         : PIN_MEMORY,
    # Output
    "OUTPUT_DIR"         : OUTPUT_DIR,
    "CHECKPOINT_DIR"     : CHECKPOINT_DIR,
}

# ── Build DataLoaders ─────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = build_dataloaders(
    data_norm, neighbor_indices, neighbor_distances, config, PRIMARY_DEVICE
)

# ── Create model ──────────────────────────────────────────────────────────────
model = create_model(config).to(PRIMARY_DEVICE)

# ── Print VRAM estimate ───────────────────────────────────────────────────────
if torch.cuda.is_available():
    n_cells, n_timesteps, _ = data_norm.shape
    est_gb = estimate_neighbor_cache_gb(n_cells, K_NEIGHBORS, n_timesteps)
    total_gb = torch.cuda.get_device_properties(PRIMARY_DEVICE).total_memory / (1024**3)
    print(f"\nVRAM estimate")
    print(f"  Neighbour cache   : {est_gb:.2f} GB")
    print(f"  Available VRAM    : {total_gb:.2f} GB")
    if est_gb > total_gb * CACHE_VRAM_FRACTION:
        print("  ⚠️  Cache may exceed VRAM budget — CACHE_NEIGHBORS will use CPU fallback.")

# ── Train ─────────────────────────────────────────────────────────────────────
model, history = train_model(
    model, train_loader, val_loader, config, PRIMARY_DEVICE, ALL_DEVICES
)

## 12 — Visualise & Evaluate

In [ ]:
# Training curves
plot_training_curves(history, OUTPUT_DIR)

# Test-set evaluation
print("\n" + "=" * 60)
print("Test-set evaluation")
print("=" * 60)
test_metrics = evaluate_on_loader(model, test_loader, PRIMARY_DEVICE, USE_AMP)

# Prediction examples
print("\nPlotting prediction examples ...")
plot_predictions(
    model, test_loader, PRIMARY_DEVICE,
    scalers, OUTPUT_DIR,
    n_samples=4, use_amp=USE_AMP,
)

print("\n✅ All done!")
print(f"Checkpoints saved in  : {CHECKPOINT_DIR}/")
print(f"Visualisations saved in: {OUTPUT_DIR}/")